# Word2Vec Word Vectorization

In [ ]:
import glob
import os
from gensim.models import Word2Vec
import gensim
import nltk
nltk.download('punkt_tab')
from nltk.tokenize import sent_tokenize, word_tokenize

import warnings

warnings.filterwarnings(action='ignore')
import pandas as pd

## Read in the Data Files

In [ ]:
# https://archive.ics.uci.edu/dataset/311/sentence+classification 
input_folder = "../data/labeled_articles/"
file_list=glob.glob("../data/labeled_articles/*.txt")

dfs = []
for filename in file_list:
    #print(f"Processing file: {filename}")
    df = pd.read_csv(filename, delimiter='\t', names=['label', 'text'], index_col=None, header=None, on_bad_lines='skip')
    df['file'] = os.path.basename(filename)
    if df['text'].isnull().any():
        #print(f"Missing values found in file: {filename}")
        df['text']=df["label"].str[5:]  # Fix for files with missing text values
        df['label']=df["label"].str[:4]
    dfs.append(df)

df = pd.concat(dfs, ignore_index=True)
print(df.count())
df

In [ ]:

data = []
for text in df['text']:
    # clean the text
    cleaned_text = text.replace('\n', ' ').replace('\r', ' ').strip()
    # tokenize the text into sentences
    for i in sent_tokenize(cleaned_text):
        temp = []

        # tokenize the sentence into words
        for j in word_tokenize(i):
            temp.append(j.lower())
    data.append(temp)

## Build the CBOW Model

In [ ]:
cbow_model = gensim.models.Word2Vec(data, min_count=1,
                                vector_size=100, window=5)
print(cbow_model.wv.index_to_key)  # list of words in vocabulary
print(len(cbow_model.wv.index_to_key))  # number of words in vocabulary
print(cbow_model.wv['the'])    # get vector for word 'the'
print(cbow_model.wv['citation'])    # get vector for word 'citation'

## Work with similarities

In [ ]:
print(cbow_model.wv.similar_by_word('citation', topn=5) ) # get 5 most similar words to 'citation'
# calculate cosine similarity between two words 
# -1 (opposite) 
# 1 (very similar),
print("Cosine similarity between citation and research: ",
      cbow_model.wv.similarity('citation', 'research'))

print("Cosine similarity between one and two: ",
      cbow_model.wv.similarity('one', 'two'))